# Análise de Dados com Python — Módulo 1

**Pipeline:** Raw → Silver → Gold  
**Banco:** PostgreSQL  
**Execução:** VSCode + pgAdmin 4

Este notebook responde às sete perguntas de negócio da atividade. As consultas são executadas no PostgreSQL e os resultados são apresentados em tabelas e gráficos.

## 1. Conexão com o PostgreSQL

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Permite importar config.py e banco.py quando o notebook é executado no VSCode.
PASTA_PROJETO = Path.cwd()
if str(PASTA_PROJETO) not in sys.path:
    sys.path.append(str(PASTA_PROJETO))

from banco import conectar

conexao = conectar()
print("Conexão PostgreSQL estabelecida com sucesso.")

## 2. Pergunta 1 — Quais são os 5 órgãos com maior custo total?

O custo total é o campo calculado `valor_total` da camada Silver.

In [ ]:
sql_q1 = """
SELECT
    nome_orgao_superior AS orgao,
    COUNT(*) AS quantidade_viagens,
    ROUND(SUM(valor_total), 2) AS custo_total
FROM silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5;
"""
df_q1 = pd.read_sql_query(sql_q1, conexao)
df_q1

In [ ]:
ax = df_q1.plot.barh(x="orgao", y="custo_total", legend=True, figsize=(10, 6))
ax.invert_yaxis()
ax.set_title("5 órgãos com maior custo total")
ax.legend(title="Indicador")
ax.set_xlabel("Custo total (R$)")
ax.set_ylabel("Órgão")
plt.tight_layout()
plt.show()


## 3. Pergunta 2 — Quais são os 3 destinos com maior custo médio por viagem?

Para evitar contar duas vezes a mesma viagem quando ela possui mais de um trecho para o mesmo destino, a consulta considera cada par **viagem + destino** uma única vez antes de calcular a média.

In [ ]:
sql_q2 = """
WITH destinos_unicos AS (
    SELECT DISTINCT
        id_viagem,
        destino_uf,
        destino_cidade
    FROM silver_trecho
    WHERE NULLIF(TRIM(destino_cidade), '') IS NOT NULL
),
base AS (
    SELECT
        d.destino_uf,
        d.destino_cidade,
        v.id_viagem,
        v.valor_total
    FROM destinos_unicos d
    JOIN silver_viagem v ON v.id_viagem = d.id_viagem
)
SELECT
    COALESCE(destino_uf, 'Exterior/Não informado') AS uf,
    destino_cidade AS destino,
    COUNT(DISTINCT id_viagem) AS viagens,
    ROUND(AVG(valor_total), 2) AS custo_medio_por_viagem
FROM base
GROUP BY destino_uf, destino_cidade
ORDER BY custo_medio_por_viagem DESC
LIMIT 3;
"""
df_q2 = pd.read_sql_query(sql_q2, conexao)
df_q2

In [ ]:
df_q2["rotulo"] = df_q2["destino"] + " (" + df_q2["uf"] + ")"
ax = df_q2.plot.bar(x="rotulo", y="custo_medio_por_viagem", legend=True, figsize=(10, 6))
ax.set_title("3 destinos com maior custo médio por viagem")
ax.legend(title="Indicador")
ax.set_xlabel("Destino")
ax.set_ylabel("Custo médio por viagem (R$)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 4. Pergunta 3 — Qual foi a viagem de maior duração e qual seu custo total?

In [ ]:
sql_q3 = """
SELECT
    id_viagem,
    nome_viajante,
    nome_orgao_superior AS orgao,
    data_inicio,
    data_fim,
    duracao_dias,
    ROUND(valor_total, 2) AS custo_total,
    destinos
FROM silver_viagem
WHERE duracao_dias IS NOT NULL
ORDER BY duracao_dias DESC, valor_total DESC
LIMIT 1;
"""
df_q3 = pd.read_sql_query(sql_q3, conexao)
df_q3

In [ ]:
ax = df_q3.plot.bar(x="id_viagem", y="custo_total", legend=True, figsize=(9, 5))
ax.set_title("Viagem de maior duração — custo total")
ax.legend(title="Indicador")
ax.set_xlabel("ID da viagem")
ax.set_ylabel("Custo total (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


# Camada Gold

A tabela e a VIEW abaixo materializam uma visão agregada por órgão. O pagamento é primeiro agregado por viagem para evitar multiplicação de valores quando há vários registros de pagamento para uma mesma viagem. Em seguida, ocorre o `JOIN` com a Silver e o `GROUP BY` por órgão.

In [ ]:
sql_gold = """
DROP VIEW IF EXISTS vw_gold_uf_destino;
DROP TABLE IF EXISTS gold_uf_destino;
DROP VIEW IF EXISTS vw_gold_transporte;
DROP TABLE IF EXISTS gold_transporte;
DROP VIEW IF EXISTS vw_gold_tipo_pagamento;
DROP TABLE IF EXISTS gold_tipo_pagamento;
DROP VIEW IF EXISTS vw_gold_orgao_pagador;
DROP TABLE IF EXISTS gold_orgao_pagador;
DROP VIEW IF EXISTS vw_gold_orgao;
DROP TABLE IF EXISTS gold_orgao;

-- Agregação por órgão superior: custo das viagens e pagamentos associados.
CREATE TABLE gold_orgao AS
SELECT
    v.nome_orgao_superior AS orgao,
    COUNT(DISTINCT v.id_viagem) AS quantidade_viagens,
    ROUND(SUM(v.valor_total), 2) AS custo_total,
    ROUND(AVG(v.valor_total), 2) AS custo_medio_por_viagem,
    ROUND(COALESCE(SUM(p.total_pagamento_viagem), 0), 2) AS total_pagamentos,
    COALESCE(SUM(p.quantidade_pagamentos), 0) AS quantidade_pagamentos
FROM silver_viagem v
LEFT JOIN (
    SELECT id_viagem, SUM(valor) AS total_pagamento_viagem, COUNT(*) AS quantidade_pagamentos
    FROM silver_pagamento
    GROUP BY id_viagem
) p ON p.id_viagem = v.id_viagem
GROUP BY v.nome_orgao_superior;

CREATE INDEX idx_gold_orgao_custo ON gold_orgao(custo_total DESC);
CREATE VIEW vw_gold_orgao AS SELECT * FROM gold_orgao;

-- Órgão pagador efetivo: não confundir com órgão superior da viagem.
CREATE TABLE gold_orgao_pagador AS
SELECT
    COALESCE(NULLIF(TRIM(p.nome_orgao_pagador), ''), 'Não informado') AS orgao_pagador,
    COUNT(*) AS quantidade_pagamentos,
    COUNT(DISTINCT p.id_viagem) AS quantidade_viagens,
    ROUND(SUM(p.valor), 2) AS total_pago,
    ROUND(AVG(p.valor), 2) AS valor_medio_pagamento
FROM silver_pagamento p
JOIN silver_viagem v ON v.id_viagem = p.id_viagem
GROUP BY COALESCE(NULLIF(TRIM(p.nome_orgao_pagador), ''), 'Não informado');

CREATE INDEX idx_gold_orgao_pagador_total ON gold_orgao_pagador(total_pago DESC);
CREATE VIEW vw_gold_orgao_pagador AS SELECT * FROM gold_orgao_pagador;

-- Três agregações adicionais para responder às perguntas 4, 5 e 6 pela Gold.
DROP VIEW IF EXISTS vw_gold_tipo_pagamento;
DROP TABLE IF EXISTS gold_tipo_pagamento;
CREATE TABLE gold_tipo_pagamento AS
SELECT tipo_pagamento, COUNT(*) AS quantidade_pagamentos,
       ROUND(AVG(valor), 2) AS valor_medio
FROM silver_pagamento
WHERE NULLIF(TRIM(tipo_pagamento), '') IS NOT NULL
GROUP BY tipo_pagamento;
CREATE VIEW vw_gold_tipo_pagamento AS SELECT * FROM gold_tipo_pagamento;

DROP VIEW IF EXISTS vw_gold_transporte;
DROP TABLE IF EXISTS gold_transporte;
CREATE TABLE gold_transporte AS
SELECT TRIM(meio_transporte) AS meio_transporte, COUNT(*) AS quantidade_trechos
FROM silver_trecho
WHERE NULLIF(TRIM(meio_transporte), '') IS NOT NULL
GROUP BY TRIM(meio_transporte);
CREATE VIEW vw_gold_transporte AS SELECT * FROM gold_transporte;

DROP VIEW IF EXISTS vw_gold_uf_destino;
DROP TABLE IF EXISTS gold_uf_destino;
CREATE TABLE gold_uf_destino AS
SELECT TRIM(destino_uf) AS uf_destino, COUNT(*) AS quantidade_trechos
FROM silver_trecho
WHERE NULLIF(TRIM(destino_uf), '') IS NOT NULL
GROUP BY TRIM(destino_uf);
CREATE VIEW vw_gold_uf_destino AS SELECT * FROM gold_uf_destino;
"""
with conexao.cursor() as cursor:
    cursor.execute(sql_gold)
conexao.commit()
print("Tabelas e VIEWs Gold criadas com sucesso.")

## 5. Pergunta 4 — Qual tipo de pagamento apresenta o maior valor médio?

A resposta utiliza a VIEW agregada `vw_gold_tipo_pagamento`, criada na camada Gold por meio de `GROUP BY`.

In [ ]:
sql_q4 = """
SELECT tipo_pagamento, quantidade_pagamentos, valor_medio
FROM vw_gold_tipo_pagamento
ORDER BY valor_medio DESC, tipo_pagamento
LIMIT 1;
"""
df_q4 = pd.read_sql_query(sql_q4, conexao)
df_q4

In [ ]:
ax = df_q4.plot.bar(x="tipo_pagamento", y="valor_medio", legend=True, figsize=(9, 5))
ax.set_title("Tipo de pagamento com maior valor médio")
ax.legend(title="Indicador")
ax.set_xlabel("Tipo de pagamento")
ax.set_ylabel("Valor médio (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Pergunta 5 — Qual é o meio de transporte mais usado nos trechos?

A resposta utiliza a VIEW agregada `vw_gold_transporte`, criada na camada Gold por meio de `GROUP BY`.

In [ ]:
sql_q5 = """
SELECT meio_transporte, quantidade_trechos
FROM vw_gold_transporte
ORDER BY quantidade_trechos DESC, meio_transporte
LIMIT 1;
"""
df_q5 = pd.read_sql_query(sql_q5, conexao)
df_q5

In [ ]:
ax = df_q5.plot.bar(x="meio_transporte", y="quantidade_trechos", legend=True, figsize=(9, 5))
ax.set_title("Meio de transporte mais usado nos trechos")
ax.legend(title="Indicador")
ax.set_xlabel("Meio de transporte")
ax.set_ylabel("Quantidade de trechos")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Pergunta 6 — Qual UF de destino aparece em mais trechos?

A resposta utiliza a VIEW agregada `vw_gold_uf_destino`, criada na camada Gold por meio de `GROUP BY`.

In [ ]:
sql_q6 = """
SELECT uf_destino, quantidade_trechos
FROM vw_gold_uf_destino
ORDER BY quantidade_trechos DESC, uf_destino
LIMIT 1;
"""
df_q6 = pd.read_sql_query(sql_q6, conexao)
df_q6

In [ ]:
ax = df_q6.plot.bar(x="uf_destino", y="quantidade_trechos", legend=True, figsize=(9, 5))
ax.set_title("UF de destino com maior quantidade de trechos")
ax.legend(title="Indicador")
ax.set_xlabel("UF de destino")
ax.set_ylabel("Quantidade de trechos")
plt.tight_layout()
plt.show()


## 8. Pergunta 7 — Qual órgão pagou mais no total?

A resposta utiliza a agregação `gold_orgao_pagador`, que soma os valores de `silver_pagamento` por `nome_orgao_pagador`. Isso distingue o órgão pagador efetivo do órgão superior responsável pela viagem. A consulta é feita pela VIEW Gold criada com `JOIN` e `GROUP BY`.


In [ ]:
sql_q7 = """
SELECT
    orgao_pagador AS orgao,
    quantidade_viagens,
    quantidade_pagamentos,
    total_pago
FROM vw_gold_orgao_pagador
ORDER BY total_pago DESC, orgao_pagador
LIMIT 1;
"""
df_q7 = pd.read_sql_query(sql_q7, conexao)
df_q7

In [ ]:
ax = df_q7.plot.bar(x="orgao", y="total_pago", legend=True, figsize=(10, 5))
ax.set_title("Órgão com maior total pago")
ax.legend(title="Indicador")
ax.set_xlabel("Órgão")
ax.set_ylabel("Total pago (R$)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 9. Visão consolidada da Gold

A consulta abaixo permite conferir as principais métricas agregadas por órgão.

In [ ]:
pd.read_sql_query("""
SELECT *
FROM vw_gold_orgao
ORDER BY custo_total DESC
LIMIT 10;
""", conexao)

## 10. Interpretação dos resultados e fechamento

Após executar todas as células, registre aqui as conclusões com base nos resultados exibidos. Não preencha com valores estimados: use os resultados efetivamente retornados pelo PostgreSQL.

- **Concentração de custos:** compare os cinco órgãos líderes e observe a diferença entre o primeiro colocado e os demais.
- **Destinos:** identifique se os maiores custos médios decorrem de poucos casos ou de volume recorrente; a consulta mostra a quantidade de viagens por destino.
- **Duração:** confira as datas de início/fim e o custo total da viagem de maior duração.
- **Pagamentos:** compare valor médio por tipo de pagamento e total pago por órgão pagador efetivo; são métricas distintas.
- **Mobilidade:** relacione o meio de transporte mais frequente à distribuição dos trechos por UF de destino.

- A camada **Raw** preserva os quatro CSVs sem tipagem ou tratamento de conteúdo.
- A camada **Silver** converte datas e valores, calcula `valor_total` e `duracao_dias` e aplica PK, FK e constraints.
- A camada **Gold** consolida indicadores por órgão superior, órgão pagador, tipo de pagamento, meio de transporte e UF de destino. As agregações usam `JOIN` + `GROUP BY` quando há relação entre tabelas e disponibilizam tabelas e VIEWs para reutilização.
- As sete perguntas de negócio são respondidas com SQL, tabelas e gráficos no próprio notebook.


In [ ]:
conexao.close()
print("Conexão encerrada.")